# RAG (Retrieval-Augmented Generation) - Chatbot Đơn Giản

## Giới thiệu về RAG

**RAG** là gì?
- RAG kết hợp **Retrieval** (Truy xuất thông tin) và **Generation** (Sinh văn bản)
- Giúp LLM trả lời câu hỏi dựa trên dữ liệu riêng của bạn
- Giảm hiện tượng "hallucination" (LLM bịa đặt thông tin)

**Quy trình RAG:**
1. **Indexing**: Chia văn bản thành chunks → Tạo embeddings → Lưu vào vector database
2. **Retrieval**: Nhận câu hỏi → Tìm chunks liên quan nhất
3. **Generation**: Đưa chunks + câu hỏi vào LLM → Sinh câu trả lời

**Demo này sẽ xây dựng:**
- ✅ RAG đơn giản với dữ liệu về Python
- ✅ Vector database đơn giản (không cần server)
- ✅ Chatbot có thể trả lời câu hỏi về Python

In [2]:
# Cài đặt các thư viện cần thiết
%pip install chromadb tiktoken sentence-transformers chromadb groq python-dotenv langchain langchain-text-splitters rank-bm25 pypdf python-docx ipywidgets -q

print("Đã cài đặt thành công!")

Note: you may need to restart the kernel to use updated packages.
Đã cài đặt thành công!



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Import các thư viện
import os
from typing import List, Tuple
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import numpy as np
from pathlib import Path
import glob

# For reading different file formats
from pypdf import PdfReader
from docx import Document

print("Import thành công!")

Import thành công!


## Bước 1: Chuẩn bị dữ liệu (Knowledge Base)

**2 cách để chuẩn bị documents:**
1. **Upload files** - Đọc từ .txt, .pdf, .docx, .md (RECOMMENDED)
2. **Manual input** - Hardcode text trong code (for testing)

Chúng ta sẽ implement cả 2 cách!

In [4]:
# Functions để đọc file documents
def load_document(file_path: str) -> str:
    """Đọc document từ file (.txt, .pdf, .docx, .md)"""
    from pathlib import Path
    
    file_ext = Path(file_path).suffix.lower()
    
    try:
        if file_ext == '.txt' or file_ext == '.md':
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
                
        elif file_ext == '.pdf':
            reader = PdfReader(file_path)
            text = ""
            for page in reader.pages:
                text += page.extract_text() + "\n"
            return text
            
        elif file_ext == '.docx':
            doc = Document(file_path)
            return "\n".join([p.text for p in doc.paragraphs])
            
        else:
            print(f"Unsupported format: {file_ext}")
            return ""
    except Exception as e:
        print(f"Lỗi: {str(e)}")
        return ""

def load_documents_from_folder(folder_path: str) -> List[str]:
    """Load tất cả documents từ folder"""
    from pathlib import Path
    
    folder = Path(folder_path)
    if not folder.exists():
        print(f"Folder không tồn tại: {folder_path}")
        return []
    
    documents = []
    file_patterns = ['*.txt', '*.pdf', '*.docx', '*.md']
    
    # Tìm tất cả files
    all_files = []
    for pattern in file_patterns:
        all_files.extend(folder.glob(pattern))
    
    print(f"Tìm thấy {len(all_files)} files")
    
    # Đọc từng file
    for file_path in all_files:
        content = load_document(str(file_path))
        if content.strip():
            documents.append(content)
            print(f"  {file_path.name} ({len(content)} chars)")
    
    return documents

print("File readers sẵn sàng!")

File readers sẵn sàng!


### Option 1: Load documents từ folder 📂

Nếu bạn đã có folder chứa files, load tất cả cùng lúc.

### Option 0: Upload files trực tiếp 📤 (EASIEST!)

Click nút bên dưới để chọn file từ máy tính của bạn!

In [ ]:
from ipywidgets import FileUpload, Button, VBox, Output
from IPython.display import display
import io

# Tạo widget upload file
file_uploader = FileUpload(
    accept='.txt,.pdf,.docx,.md',  # Chỉ chấp nhận các định dạng này
    multiple=True,  # Cho phép chọn nhiều file
    description='Chọn files'
)

# Output area để hiển thị kết quả
output = Output()

# Biến lưu documents đã upload
uploaded_documents = []

def process_uploaded_files(change):
    """Xử lý khi user upload files"""
    global uploaded_documents
    uploaded_documents = []
    
    with output:
        output.clear_output()
        
        if not file_uploader.value:
            print("Chưa có file nào được chọn")
            return
        
        print(f"Đang xử lý {len(file_uploader.value)} file(s)...\n")
        
        for uploaded_file in file_uploader.value:
            filename = uploaded_file['name']
            content = uploaded_file['content']
            
            try:
                # Xác định loại file
                if filename.endswith('.txt') or filename.endswith('.md'):
                    text = content.decode('utf-8')
                    
                elif filename.endswith('.pdf'):
                    
                    pdf_file = io.BytesIO(content)
                    reader = PdfReader(pdf_file)
                    text = ""
                    for page in reader.pages:
                        text += page.extract_text() + "\n"
                        
                elif filename.endswith('.docx'):
                    docx_file = io.BytesIO(content)
                    doc = Document(docx_file)
                    text = "\n".join([p.text for p in doc.paragraphs])
                else:
                    print(f"{filename}: Định dạng không hỗ trợ")
                    continue
                
                if text.strip():
                    uploaded_documents.append(text)
                    print(f"{filename} ({len(text)} chars)")
                else:
                    print(f"{filename}: File trống")
                    
            except Exception as e:
                print(f"{filename}: Lỗi - {str(e)}")
        
        if uploaded_documents:
            print(f"\nĐã upload thành công {len(uploaded_documents)} document(s)!")
            print(f"   Tổng: {sum(len(d) for d in uploaded_documents):,} ký tự")
        else:
            print("\nKhông có document nào được load thành công")

# Gắn event handler
file_uploader.observe(process_uploaded_files, names='value')

# Hiển thị widget
print("UPLOAD FILES TỪ MÁY TÍNH")
print("="*50)
print("Hỗ trợ: .txt, .pdf, .docx, .md")
print("Có thể chọn nhiều files cùng lúc\n")

display(VBox([file_uploader, output]))

UPLOAD FILES TỪ MÁY TÍNH
Hỗ trợ: .txt, .pdf, .docx, .md
Có thể chọn nhiều files cùng lúc



In [8]:
# Sử dụng documents đã upload (nếu có)
if uploaded_documents and len(uploaded_documents) > 0:
    raw_documents = uploaded_documents
    print(f"Sử dụng {len(uploaded_documents)} document(s) đã upload")
    print(f"   Tổng: {sum(len(d) for d in raw_documents):,} ký tự")
else:
    print("Chưa có file upload. Chạy cell phía trên để upload file!")
    print("   Hoặc tiếp tục với Option 1 (load từ folder) bên dưới")
    raw_documents = None

Sử dụng 7 document(s) đã upload
   Tổng: 34,910 ký tự


## Bước 1.5: Text Chunking - Chia văn bản thành chunks

**Tại sao cần chunking?**
- Documents thường rất dài, vượt quá context window của LLM
- Chunking giúp tìm kiếm chính xác hơn (tìm đoạn cụ thể thay vì cả tài liệu)
- Giảm nhiễu thông tin khi retrieve

**Các tham số quan trọng:**
- `chunk_size`: Kích thước mỗi chunk (số ký tự)
- `chunk_overlap`: Số ký tự chồng lấn giữa các chunks (giúp không mất context ở ranh giới)

**Sử dụng LangChain RecursiveCharacterTextSplitter:**
- Thư viện chuyên nghiệp cho chunking
- Tự động cắt thông minh theo thứ tự: \n\n → \n → . → space
- Không cần viết code thủ công

## Chia chunk với RecursiveCharacterTextSplitter

💡 **Muốn tìm hiểu sâu hơn về Chunking?**  
→ Xem file `embedding.ipynb` để so sánh chi tiết các cách chunking khác nhau!

In [9]:
# Sử dụng LangChain RecursiveCharacterTextSplitter để chia chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Khởi tạo text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,           # Kích thước mỗi chunk (ký tự)
    chunk_overlap=200,         # Số ký tự overlap giữa các chunks
    separators=["\n\n", "\n", ". ", " ", ""],  # Thứ tự ưu tiên khi cắt
)

print("Đã khởi tạo RecursiveCharacterTextSplitter")
print(f"  - Chunk size: 1000 ký tự")
print(f"  - Chunk overlap: 200 ký tự")

# Chia documents thành chunks
print("\nĐang chia documents thành chunks...")
chunked_docs = text_splitter.create_documents(raw_documents)

# Chuyển thành list string
documents = [doc.page_content for doc in chunked_docs]
print(f"Đã chia thành {len(documents)} chunks")

Đã khởi tạo RecursiveCharacterTextSplitter
  - Chunk size: 1000 ký tự
  - Chunk overlap: 200 ký tự

Đang chia documents thành chunks...
Đã chia thành 45 chunks


In [10]:
print(f"\nKết quả:")
print(f"  - {len(raw_documents)} documents gốc -> {len(documents)} chunks")
print(f"  - Trung bình {len(documents)/len(raw_documents):.1f} chunks/document")

# Hiển thị ví dụ
print(f"\nVí dụ 2 chunks đầu tiên:")
for i in range(min(2, len(documents))):
    print(f"\n{i+1}. ({len(documents[i])} chars)")
    print(f"{'─'*70}")
    print(documents[i])


Kết quả:
  - 7 documents gốc -> 45 chunks
  - Trung bình 6.4 chunks/document

Ví dụ 2 chunks đầu tiên:

1. (917 chars)
──────────────────────────────────────────────────────────────────────
FUNDAMENTALS OF PYTHON
(LẬP TRÌNH PYTHON CƠ BẢN)
Thời gian: 5 tuần
Thời lượng: 40 giờ
Học phí: 2.500.000 đ
MỤC TIÊU
Khóa học cung cấp cho học viên (HV) những kiến thức nền tảng và những kỹ năng cần thiết để có thể xây dựng các ứng dụng đơn giản bằng Python – một ngôn ngữ lập trình cấp cao, thông dịch, hướng đối tượng và đa mục đích. 
Rèn luyện và phát triển kỹ năng lập trình, tư duy logic.
Xây dựng nền tảng cơ bản vững chắc trong ngôn ngữ lập trình Python tạo tiền đề cho việc học các kiến thức lập trình Web, Game, Desktop, Machine Learning, Data Science… phát triển nghề nghiệp.
Là khóa học đầu tiên trong chương trình “Data Science & Machine Learning Certificate”
ĐỐI TƯỢNG HỌC
Sinh viên các trường Đại học, Cao đẳng
HV có định hướng sẽ làm việc với Python để xây dựng ứng dụng/ Machine Learning/ Data 

## Bước 2: Tạo Embeddings

**Embeddings** là biểu diễn vector của văn bản. Văn bản tương tự sẽ có vectors gần nhau trong không gian vector.

Sử dụng **SentenceTransformer** để tạo embeddings.

In [11]:
# Load mô hình embedding
# Sử dụng model nhỏ và nhanh cho demo
print("Đang tải embedding model...")
embedding_model = SentenceTransformer('AITeamVN/Vietnamese_Embedding_v2')
print("Đã tải model thành công!")

# Tạo embeddings cho documents
print("\nĐang tạo embeddings cho documents...")
document_embeddings = embedding_model.encode(documents, show_progress_bar=True)

print(f"\nĐã tạo {len(document_embeddings)} embeddings")
print(f"Mỗi embedding có chiều: {document_embeddings[0].shape}")

Đang tải embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Đã tải model thành công!

Đang tạo embeddings cho documents...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Đã tạo 45 embeddings
Mỗi embedding có chiều: (1024,)


## Bước 3: Lưu trữ vào Vector Database

Sử dụng **ChromaDB** - một vector database đơn giản, không cần server.

In [12]:
# Khởi tạo ChromaDB client
# Lưu trên Ram
# chroma_client = chromadb.Client()

# Lưu trên ổ cứng
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Tạo hoặc lấy collection
try:
    collection = chroma_client.create_collection(name="python_knowledge")
    print("Đã tạo collection mới")
except:
    chroma_client.delete_collection(name="python_knowledge")
    collection = chroma_client.create_collection(name="python_knowledge")
    print("Đã xóa và tạo lại collection")

# Thêm documents vào collection
collection.add(
    documents=documents,
    embeddings=document_embeddings.tolist(),
    ids=[f"doc_{i}" for i in range(len(documents))]
)

print(f"Đã thêm {len(documents)} documents vào vector database")
print(f"Collection có {collection.count()} items")

Đã tạo collection mới
Đã thêm 45 documents vào vector database
Collection có 45 items


## Bước 4: Retrieval - Tìm kiếm thông tin liên quan

Khi có câu hỏi, ta sẽ:
1. Chuyển câu hỏi thành embedding
2. Tìm các documents có embedding gần nhất (similarity search)
3. Trả về top-k documents liên quan nhất

In [10]:
def retrieve_relevant_docs(query: str, top_k: int = 3):
    """
    Tìm top-k documents liên quan nhất với câu hỏi
    
    Args:
        query: Câu hỏi của user
        top_k: Số lượng documents trả về
    
    Returns:
        List của documents liên quan
    """
    # Tạo embedding cho query
    query_embedding = embedding_model.encode([query])[0]
    
    # Tìm kiếm trong vector database
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )
    
    return results['documents'][0]



In [11]:
# Test thử retrieval
test_query = "khóa python cơ bản học phí bao nhiêu?"
print(f"Câu hỏi: {test_query}\n")
print("=" * 60)
print("Documents liên quan:")
print("=" * 60)

retrieved_docs = retrieve_relevant_docs(test_query, top_k=7)
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n{i}. {doc}")

Câu hỏi: khóa python cơ bản học phí bao nhiêu?

Documents liên quan:

1. FUNDAMENTALS OF PYTHON
(LẬP TRÌNH PYTHON CƠ BẢN)
Thời gian: 5 tuần
Thời lượng: 40 giờ
Học phí: 2.500.000 đ
MỤC TIÊU
Khóa học cung cấp cho học viên (HV) những kiến thức nền tảng và những kỹ năng cần thiết để có thể xây dựng các ứng dụng đơn giản bằng Python – một ngôn ngữ lập trình cấp cao, thông dịch, hướng đối tượng và đa mục đích. 
Rèn luyện và phát triển kỹ năng lập trình, tư duy logic.
Xây dựng nền tảng cơ bản vững chắc trong ngôn ngữ lập trình Python tạo tiền đề cho việc học các kiến thức lập trình Web, Game, Desktop, Machine Learning, Data Science… phát triển nghề nghiệp.
Là khóa học đầu tiên trong chương trình “Data Science & Machine Learning Certificate”
ĐỐI TƯỢNG HỌC
Sinh viên các trường Đại học, Cao đẳng
HV có định hướng sẽ làm việc với Python để xây dựng ứng dụng/ Machine Learning/ Data Science
Tất cả những ai muốn tìm hiểu về lập trình Python để ứng dụng trong công việc, học tập và cuộc sống.

2. Machi

## Bước 4.5: Reranking - Cải thiện chất lượng Retrieval

**Tại sao cần Reranking?**
- Vector similarity không phải lúc nào cũng chọn đúng documents tốt nhất
- Reranking giúp sắp xếp lại kết quả dựa trên nhiều signals khác nhau
- Tăng precision (độ chính xác) của retrieval

**Các phương pháp Reranking:**
1. **Cross-Encoder** - Model chuyên dụng cho reranking (chính xác nhất)
2. **BM25** - Lexical matching (complement cho semantic search)
3. **Hybrid** - Kết hợp cả semantic + lexical

Chúng ta sẽ implement cả 3 cách!

In [12]:
# Load Cross-Encoder model cho reranking
print("Đang tải Cross-Encoder model cho reranking...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Đã tải Cross-Encoder thành công!")
print("  Model: ms-marco-MiniLM-L-6-v2 (optimized for semantic reranking)")

# Chuẩn bị BM25 index
print("\nĐang chuẩn bị BM25 index...")
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)
print("BM25 index đã sẵn sàng!")
print("  BM25: Lexical matching (keyword-based)")

Đang tải Cross-Encoder model cho reranking...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Đã tải Cross-Encoder thành công!
  Model: ms-marco-MiniLM-L-6-v2 (optimized for semantic reranking)

Đang chuẩn bị BM25 index...
BM25 index đã sẵn sàng!
  BM25: Lexical matching (keyword-based)


In [13]:
# Reranking Functions
def rerank_with_cross_encoder(query: str, docs: List[str], top_k: int = 3):
    """Rerank bằng Cross-Encoder (semantic)"""
    pairs = [[query, doc] for doc in docs]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

def rerank_with_bm25(query: str, docs: List[str], top_k: int = 3):
    """Rerank bằng BM25 (lexical/keywords)"""
    tokenized_query = query.lower().split()
    doc_scores = []
    for doc in docs:
        idx = documents.index(doc) if doc in documents else -1
        score = bm25.get_scores(tokenized_query)[idx] if idx >= 0 else 0
        doc_scores.append((doc, score))
    ranked = sorted(doc_scores, key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

def rerank_hybrid(query: str, docs: List[str], top_k: int = 3, alpha: float = 0.7):
    """Hybrid reranking: semantic + lexical"""
    # Cross-Encoder scores
    ce_scores = cross_encoder.predict([[query, d] for d in docs])
    ce_norm = (ce_scores - ce_scores.min()) / (ce_scores.max() - ce_scores.min() + 1e-10)
    
    # BM25 scores
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-10)
    
    # Combine
    hybrid_scores = []
    for i, doc in enumerate(docs):
        idx = documents.index(doc) if doc in documents else i
        semantic = ce_norm[i]
        lexical = bm25_norm[idx] if idx < len(bm25_norm) else 0
        score = alpha * semantic + (1 - alpha) * lexical
        hybrid_scores.append((doc, score))
    
    ranked = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

print("Reranking functions ready")

Reranking functions ready


In [15]:
# Enhanced Retrieval với Reranking
def retrieve_and_rerank(query: str, initial_k: int = 10, final_k: int = 3, 
                       method: str = 'hybrid') -> List[str]:
    """Retrieve + Rerank documents"""
    # Step 1: Get candidates
    query_embedding = embedding_model.encode([query])[0]
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=initial_k
    )
    candidates = results['documents'][0]
    
    # Step 2: Rerank
    if method == 'cross_encoder':
        reranked = rerank_with_cross_encoder(query, candidates, final_k)
    elif method == 'bm25':
        reranked = rerank_with_bm25(query, candidates, final_k)
    else:  # hybrid
        reranked = rerank_hybrid(query, candidates, final_k)
    
    return [doc for doc, score in reranked]

print("Retrieve & Rerank ready!")

Retrieve & Rerank ready!


In [16]:
# So sánh: Không Reranking vs Có Reranking
test_query = "khóa machine learning with python mục tiêu là gì?"
print(f"Câu hỏi: {test_query}\n")

# Không reranking (baseline)
print("KHÔNG RERANKING (Baseline):")
baseline_docs = retrieve_relevant_docs(test_query, top_k=7)
for i, doc in enumerate(baseline_docs, 1):
    print(f"{i}. {doc}")

# Có reranking (hybrid)
print("\nCÓ RERANKING (Hybrid):")
reranked_docs = retrieve_and_rerank(test_query, method='hybrid')
for i, doc in enumerate(reranked_docs, 1):
    print(f"{i}. {doc}")

print("\nReranking giúp sắp xếp documents chính xác hơn!")

Câu hỏi: khóa machine learning with python mục tiêu là gì?

KHÔNG RERANKING (Baseline):
1. MỤC TIÊU
Data Science & Machine Learning (Khoa học dữ liệu & Máy học) được xếp hạng là một trong những ngành nghề “hot” nhất trong Cách mạng công nghiệp 4.0 vì thế nhu cầu về nguồn nhân lực trong lĩnh vực này đang bùng nổ với các vị trí như Data Engineer, Data Analyst, Data Scientist, AI/Machine Learning Engineer… Đây là một lĩnh vực mới và thú vị, đòi hỏi các cá nhân phải trang bị kiến thức và kỹ năng để giải quyết các vấn đề tiên tiến. 
Để đáp ứng các yêu cầu công việc này, Trung Tâm Tin Học triển khai chương trình “Data Science and Machine Learning Certificate”, với 6 khóa học như: Fundamentals of Python; Data Manipulation and Visualization with Python; Database SQL and Data Collection for Data Science; Data Pre-processing and Analysis, Machine Learning with Python; Big Data in Machine Learning và 1 đồ án tốt nghiệp.
Mục tiêu của chương trình:
Cung cấp kiến thức nền tảng và kỹ năng lập trình P

### 🎯 Khi nào dùng Reranking method nào?

**Hybrid (Recommended)**: Tốt nhất cho hầu hết trường hợp
- Balance giữa semantic và keyword matching
- Default alpha=0.7 (70% semantic, 30% lexical)

**Cross-Encoder**: Khi cần accuracy cao nhất
- Hiểu ngữ nghĩa sâu
- Chậm hơn nhưng chính xác hơn

**BM25**: Khi query có keywords cụ thể
- Nhanh nhất
- Tốt cho tài liệu kỹ thuật

## Bước 5: Generation - Tạo câu trả lời

Có 2 cách để generate câu trả lời:

### Sử dụng LLM API (Google Gemini, OpenAI, Anthropic, etc.)
- Cần API key
- Chi phí phát sinh (hoặc miễn phí với Gemini)
- Chất lượng cao

### 5. LLM-based Generation (Chuyên nghiệp - Cần API Key)

Sử dụng Groq API để tạo câu trả lời tự nhiên và thông minh hơn.

**Cách lấy Groq API Key (Miễn phí):**

1. Truy cập: https://console.groq.com/keys
2. Đăng nhập / Đăng ký tài khoản
3. Click "Create API Key"
4. Copy API key
5. Tạo file `.env` trong thư mục project với nội dung:
   ```
   GROQ_API_KEY=your_api_key_here
   ```

**Ưu điểm của Groq:**
- ✅ Miễn phí (rate limit cao)
- ✅ Tốc độ inference cực nhanh (LPU)
- ✅ Hỗ trợ nhiều models
- ✅ API đơn giản, tương thích OpenAI

In [ ]:
# Cấu hình Groq API
# Tạo file .env và thêm: GROQ_API_KEY=your_api_key_here
# Lấy API key tại: https://console.groq.com/keys

from groq import Groq
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Kiểm tra API key
api_key = os.getenv("GROQ_API_KEY")
if api_key:
    client = Groq(api_key=api_key)
    MODEL_NAME = "openai/gpt-oss-20b"
    print("Groq API key đã được cấu hình!")
    print(f"Sử dụng model: {MODEL_NAME}")
else:
    print("Chưa có Groq API key. Tạo file .env và thêm GROQ_API_KEY=your_key")
    print("Lấy API key tại: https://console.groq.com/keys")

✓ Groq API key đã được cấu hình!
✓ Sử dụng model: openai/gpt-oss-20b


In [ ]:
def advanced_rag_chatbot(query: str, use_reranking: bool = True) -> str:
    """RAG Chatbot với Groq"""
    if not api_key:
        return "Cần Groq API key!"
    
    # Retrieve + Rerank
    docs = retrieve_and_rerank(query) if use_reranking else retrieve_relevant_docs(query)
    context = "\n\n".join(docs)
    
    # Generate with LLM
    prompt = f"""Trả lời câu hỏi dựa trên CONTEXT.

CONTEXT:
{context}

QUESTION: {query}

Trả lời lịch sự bằng tiếng Việt. Nếu không có thông tin, nói "Tôi không tìm thấy trong tài liệu"."""
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Bạn là trợ lý AI hữu ích, trả lời bằng tiếng Việt."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=1024
    )
    
    return response.choices[0].message.content

In [ ]:
# Test
test_query = "khóa machine learning with python đối tượng học là ai?"
print(f"Câu hỏi: {test_query}\n")
print(advanced_rag_chatbot(test_query))

❓ khóa machine learning with python đối tượng học là ai?

**Đối tượng học của khóa “Machine Learning with Python”**  
Khóa học này dành cho những học viên đã tham gia khóa “Fundamentals of Python” (hoặc có kiến thức tương đương) và đang có định hướng phát triển sự nghiệp trong lĩnh vực Machine Learning, Data Science hoặc Data Analysis.
**Đối tượng học của khóa “Machine Learning with Python”**  
Khóa học này dành cho những học viên đã tham gia khóa “Fundamentals of Python” (hoặc có kiến thức tương đương) và đang có định hướng phát triển sự nghiệp trong lĩnh vực Machine Learning, Data Science hoặc Data Analysis.


## Bước 6: Demo Interactive - Thử các câu hỏi khác nhau

Bây giờ bạn có thể thử hỏi chatbot bất kỳ câu hỏi nào về Python!

In [ ]:
# Thử câu hỏi của riêng bạn!
your_question = "python có mấy kiểu dữ liệu?"  # Thay đổi câu hỏi ở đây

print("=" * 70)
print("CÂU HỎI CỦA BẠN")
print("=" * 70)
print(f"\n{your_question}\n")

# Thử Advanced RAG với Groq
if api_key:
    print("\n" + "=" * 70)
    print("Câu trả lời (Advanced RAG với Groq):")
    print("-" * 70)
    advanced_answer = advanced_rag_chatbot(your_question)
    print(advanced_answer)

CÂU HỎI CỦA BẠN

❓ python có mấy kiểu dữ liệu?


🤖 Câu trả lời (Advanced RAG với Groq):
----------------------------------------------------------------------
Python có **ba** kiểu dữ liệu cơ bản chính:  

1. **Số** (int, float)  
2. **Chuỗi** (str)  
3. **Boolean** (bool)  

Ngoài ra, Python còn hỗ trợ các kiểu dữ liệu tổ hợp như list, tuple, set, dictionary.
Python có **ba** kiểu dữ liệu cơ bản chính:  

1. **Số** (int, float)  
2. **Chuỗi** (str)  
3. **Boolean** (bool)  

Ngoài ra, Python còn hỗ trợ các kiểu dữ liệu tổ hợp như list, tuple, set, dictionary.


## 🎯 Tổng kết

### Bạn đã học được gì?

1. **Load Documents**: Từ file hoặc hardcode
2. **Chunking**: Chia nhỏ documents
3. **Embedding**: Chuyển text thành vectors
4. **Vector Store**: Lưu trữ với ChromaDB
5. **Retrieval**: Tìm documents liên quan
6. **Reranking**: Sắp xếp lại chính xác hơn (3 methods)
7. **Generation**: Tạo câu trả lời với LLM

### RAG Workflow:
```
Documents → Chunk → Embed → Store → Retrieve → Rerank → Generate
```

### Key Benefits:
✅ LLM có kiến thức mới nhất  
✅ Giảm hallucination  
✅ Tiết kiệm chi phí vs fine-tuning  
✅ Reranking cải thiện 20-30% accuracy

## 🚀 Bài tập cho học viên

### Bài 1: Mở rộng Knowledge Base
- Thêm documents mới về chủ đề khác
- Load từ file .txt, .pdf hoặc .docx
- Test với tài liệu tiếng Việt

### Bài 2: Thử nghiệm Reranking
- Thay đổi `alpha` trong hybrid (0-1)
- So sánh 3 methods với các câu hỏi khác nhau
- Chọn method phù hợp với use case của bạn

### Bài 3: Cải thiện Chatbot
- Thêm conversation history (nhớ context)
- Tạo UI với Streamlit
- Thêm citation (hiện nguồn tài liệu)

## 🎁 Bonus: Complete RAG Class để tái sử dụng

In [ ]:
class SimpleRAGChatbot:
    """RAG Chatbot đơn giản - dễ tái sử dụng"""
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        print("Khởi tạo RAG Chatbot...")
        self.embedding_model = SentenceTransformer(model_name)
        self.chroma_client = chromadb.Client()
        
        try:
            self.collection = self.chroma_client.create_collection("rag_demo")
        except:
            self.chroma_client.delete_collection("rag_demo")
            self.collection = self.chroma_client.create_collection("rag_demo")
        
        print("Chatbot sẵn sàng!")
    
    def add_documents(self, docs: List[str]):
        """Thêm documents vào knowledge base"""
        embeddings = self.embedding_model.encode(docs)
        self.collection.add(
            documents=docs,
            embeddings=embeddings.tolist(),
            ids=[f"doc_{i}" for i in range(len(docs))]
        )
        print(f"Đã thêm {len(docs)} documents")
    
    def chat(self, query: str, top_k: int = 3) -> str:
        """Trả lời câu hỏi"""
        # Retrieve
        query_emb = self.embedding_model.encode([query])[0]
        results = self.collection.query(
            query_embeddings=[query_emb.tolist()],
            n_results=top_k
        )
        context = "\n\n".join(results['documents'][0])
        
        # Generate (simple template)
        return f"""Câu trả lời:

{context}

Tip: Integrate với Gemini API để có câu trả lời tự nhiên hơn!
"""

# Demo
print("=" * 50)
bot = SimpleRAGChatbot()
bot.add_documents(documents)
print(bot.chat("NumPy là gì?"))